In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("metadata_v1.csv")
df.head()

,id,video_id,image_id,context_id,ocr,timestamp,video_url,fps,video_path
0,0,K01_V001,K01_V001/0,K01_V001,HIP 18:29:57 giayi FAT,00:00.000,https://youtube.com/watch?v=ls1EfyBswW8,25.0,../video/K01_V001.mp4
1,1,K01_V001,K01_V001/25,K01_V001,18:29:58 giay,00:01.000,https://youtube.com/watch?v=ls1EfyBswW8,25.0,../video/K01_V001.mp4
2,2,K01_V001,K01_V001/50,K01_V001,HD 18:29:59,00:02.000,https://youtube.com/watch?v=ls1EfyBswW8,25.0,../video/K01_V001.mp4
3,3,K01_V001,K01_V001/75,K01_V001,TV 18:30:00 giay,00:03.000,https://youtube.com/watch?v=ls1EfyBswW8,25.0,../video/K01_V001.mp4
4,4,K01_V001,K01_V001/100,K01_V001,TY HD 18:30:01 giay,00:04.000,https://youtube.com/watch?v=ls1EfyBswW8,25.0,../video/K01_V001.mp4


In [3]:
df['timestamp'].dtype

dtype('O')

#### **Bổ sung transcript K01-K20, L21-L30 crawl được từ YouTube cho AIC'25**

In [16]:
def parse_ts_to_seconds(ts: pd.Series) -> np.ndarray:
    # Ensure strings
    s = pd.Series(ts, copy=False).astype(str)

    # Pad to hh:mm:ss(.ms) if it's mm:ss(.ms)
    s = np.where(s.str.count(':') == 1, '00:' + s, s)
    s = pd.Series(s, copy=False)

    # Extract hh, mm, ss, ms (ms optional)
    # examples matched: 00:01:02.345, 00:01:02, 01:02.5 (after padding becomes 00:01:02.5)
    m = s.str.extract(r'(?:(\d+):)?(\d+):(\d+)(?:\.(\d+))?$', expand=True)
    # columns: 0=hh?, 1=mm, 2=ss, 3=ms?

    # Fill missing and convert
    hh = m[0].fillna('0').astype(float).to_numpy()
    mm = m[1].fillna('0').astype(float).to_numpy()
    ss = m[2].fillna('0').astype(float).to_numpy()

    # Normalize ms to 3 digits precision (right-pad/truncate) then to seconds
    ms_str = m[3].fillna('0')
    ms = (ms_str.str.ljust(3, '0').str[:3].astype(float) / 1000.0).to_numpy()

    return hh * 3600.0 + mm * 60.0 + ss + ms

def attach_transcripts_to_group(group_df: pd.DataFrame,
                                transcript_path: str,
                                nearest_within_sec: float = 5.0) -> pd.Series:
    """
    Vectorized core: for a single video_id partition (group_df),
    load its transcript CSV and produce the 'transcript' Series aligned to group_df.index.
    """

    # If no transcript file => return all-NaN column quickly
    if not os.path.exists(transcript_path):
        return pd.Series(index=group_df.index, dtype=object)

    # Read transcript; expected columns: text, start, duration
    transcript_df = pd.read_csv(transcript_path)
    if transcript_df.empty or not {"text", "start", "duration"}.issubset(transcript_df.columns):
        return pd.Series(index=group_df.index, dtype=object)

    # Ensure numeric & sorted
    starts = transcript_df["start"].to_numpy(dtype=float)
    durations = transcript_df["duration"].to_numpy(dtype=float)
    order = np.argsort(starts)
    starts = starts[order]
    durations = durations[order]
    ends = starts + durations
    texts = transcript_df["text"].astype(str).to_numpy()[order]

    n_seg = len(starts)
    if n_seg == 0:
        return pd.Series(index=group_df.index, dtype=object)

    # Metadata timestamps (seconds)
    t_img = parse_ts_to_seconds(group_df["timestamp"])

    # Find candidate previous segment by start time (binary search)
    # idx_prev = index of the last segment whose start <= t
    idx_prev = np.searchsorted(starts, t_img, side="right") - 1

    # Inside check
    valid_prev = (idx_prev >= 0)
    inside = np.zeros_like(idx_prev, dtype=bool)
    inside[valid_prev] = t_img[valid_prev] < ends[idx_prev[valid_prev]]

    chosen = np.full_like(idx_prev, fill_value=-1)
    chosen[inside] = idx_prev[inside]

    # Nearest-within-5s fallback
    outside = ~inside
    if np.any(outside):
        o_idx_prev = idx_prev[outside]
        o_t = t_img[outside]

        prev_exists = o_idx_prev >= 0
        dist_prev = np.full_like(o_t, fill_value=np.inf, dtype=float)
        dist_prev[prev_exists] = o_t[prev_exists] - ends[o_idx_prev[prev_exists]]

        next_idx = o_idx_prev + 1
        next_exists = next_idx < n_seg
        dist_next = np.full_like(o_t, fill_value=np.inf, dtype=float)
        dist_next[next_exists] = starts[next_idx[next_exists]] - o_t[next_exists]

        choose_prev = dist_prev <= dist_next
        min_dist = np.minimum(dist_prev, dist_next)
        accept = min_dist <= nearest_within_sec

        cand_idx = np.where(choose_prev, o_idx_prev, next_idx)
        cand_idx[~accept] = -1
        chosen[outside] = cand_idx

    # Gather [chosen-2..chosen+2]
    offsets = np.array([-2, -1, 0, 1, 2])
    m = chosen.shape[0]
    idx_matrix = chosen[np.newaxis, :] + offsets[:, np.newaxis]

    valid_idx = (idx_matrix >= 0) & (idx_matrix < n_seg)
    texts_pad = np.concatenate([texts, np.array([""], dtype=object)])
    pad_pos = len(texts)

    idx_clipped = idx_matrix.copy()
    idx_clipped[~valid_idx] = pad_pos

    gathered = texts_pad[idx_clipped]  # (5, m), object

    # gathered: shape (5, m), dtype object -> cast to Unicode
    g = gathered.astype(np.str_)  # or gathered.astype('U')

    # Vectorized join across the 5 rows (only 4 adds; tiny constant loop)
    s = g[0]
    for k in range(1, g.shape[0]):
        s = np.char.add(np.char.add(s, " "), g[k])

    # Normalize whitespace and finalize as a Series
    s = pd.Series(s, index=group_df.index, dtype="string").str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.mask(s.eq(""), None)

    return s

In [17]:
def in_target(v: str) -> bool:
    # Expect patterns like 'K01_V001', 'L22_V003', etc.
    # We check the leading letter and two-digit number.
    if not isinstance(v, str) or len(v) < 3:
        return False
    letter = v[0]
    if letter == 'K':
        try:
            num = int(v[1:3])
            return 1 <= num <= 20
        except Exception:
            return False
    if letter == 'L':
        try:
            num = int(v[1:3])
            return 21 <= num <= 30
        except Exception:
            return False
    return False

In [18]:
TRANSCRIPTS_DIR = "./transcripts"
NEAREST_WITHIN_SEC = 5.0
OUTPUT_PATH = "./metadata_v2.csv"

In [19]:
mask_target = df["video_id"].map(in_target)
df_target = df[mask_target].copy()
df_non_target = df[~mask_target].copy()

# Prepare a result series. Index is reduced to df_target's index for efficiency
transcript_col = pd.Series(index=df_target.index, dtype=object)

# Process per-video_id group to avoid N×M joins
for vid, g in df_target.groupby("video_id", sort=False):
    transcript_path = os.path.join(TRANSCRIPTS_DIR, f"{vid}.csv")
    transcript_col.loc[g.index] = attach_transcripts_to_group(
        g, transcript_path, nearest_within_sec=NEAREST_WITHIN_SEC
    )

# Stitch back
df_target["transcript"] = transcript_col
if not df_non_target.empty:
    df_non_target["transcript"] = np.nan

df2 = pd.concat([df_target, df_non_target], axis=0).sort_index()


In [20]:
# check how many transcripts were attached
num_attached = df2["transcript"].notna().sum()
print(f"Number of rows with attached transcripts: {num_attached} out of {len(df2)}")

Number of rows with attached transcripts: 1253072 out of 1323846


In [23]:
# print video_id of rows without attached transcripts
missing_transcripts = df2[df2["transcript"].isna()]["video_id"].unique()
print("Video IDs without attached transcripts:")
for vid in missing_transcripts:
    print(vid)

Video IDs without attached transcripts:
K11_V005
L24_V003
L24_V005
L24_V006
L24_V007
L24_V008
L24_V009
L24_V010
L24_V011
L24_V013
L24_V014
L24_V015
L24_V016
L24_V018
L24_V019
L24_V020
L24_V021
L24_V022
L24_V023
L24_V024
L24_V025
L24_V026
L24_V027
L24_V028
L24_V029
L24_V030
L24_V031
L24_V032
L24_V033
L24_V035
L24_V036
L24_V037
L24_V038
L24_V039
L24_V040
L24_V042
L24_V043
L24_V044
L26_V101
L26_V105
L26_V107
L26_V108
L26_V112
L26_V118
L26_V119
L26_V298
L26_V313
L26_V337
L26_V381
L26_V432
L26_V491
L28_V001
L28_V002
L28_V003
L28_V004
L28_V005
L28_V006
L28_V007
L28_V008
L28_V009
L28_V010
L28_V011
L28_V012
L28_V013
L28_V014
L28_V015
L28_V016
L28_V017
L28_V018
L28_V019
L28_V020
L28_V021
L28_V022
L28_V023
L28_V024
L30_V029
L30_V039


In [24]:
df2.to_csv(OUTPUT_PATH, index=False)

In [25]:
df2.sample(3)

,id,video_id,image_id,context_id,ocr,timestamp,video_url,fps,video_path,transcript
492521,492521,K15_V012,K15_V012/16560,K15_V012,10-05-202516:08:34 HTY9 06:43:36 áo buöc My th...,09:12.000,https://youtube.com/watch?v=RsF3IXqghC4,30.0,../video/K15_V012.mp4,phối hợp với công an tỉnh truy tìm người đàn ô...
549684,549684,K16_V030,K16_V030/30250,K16_V030,Ho ten khach hang: HTY TENMONAN TT DON LUONG T...,20:10.000,https://youtube.com/watch?v=Sl2O0yk2Apc,25.0,../video/K16_V030.mp4,Hằng cho biết do phải trích tiền hoa hùng khoả...
1285632,1285632,L29_V017,L29_V017/2500,L29_V017,HTY Dnline GlClaCaMan,01:40.000,https://youtube.com/watch?v=g7fxVyYXRts,25.0,../video/L29_V017.mp4,những nét đẹp văn hóa của người dân cuộc sống ...
